<a href="https://colab.research.google.com/github/jesusvillota/DataScience_CemfiMaster/blob/master/Session3/3_2_LLM_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="max-width: 880px; margin: 20px auto 22px; padding: 0px; border-radius: 18px; border: 1px solid #e5e7eb; background: linear-gradient(180deg, #ffffff 0%, #f9fafb 100%); box-shadow: 0 8px 26px rgba(0,0,0,0.06); overflow: hidden;">

  <!-- Banner Header -->
  <div style="padding: 34px 32px 14px; text-align: center; line-height: 1.38;">
    <div style="font-size: 13px; letter-spacing: 0.14em; text-transform: uppercase; color: #6b7280; font-weight: bold; margin-bottom: 5px;">
      Session #3 • Part 3
    </div>
    <!-- <div style="font-size: 29px; font-weight: 800; color: #14276c; margin-bottom: 4px;">
      LLMs
    </div> -->
    <div style="font-size: 26px; font-weight: 800; color: #14276c; margin-bottom: 4px;">
      Structured Data with LLMs Done Right
    </div>
    <div style="font-size: 16.5px; color: #374151; font-style: italic; margin-bottom: 0;">
      Data Science for Economics
    </div>
  </div>

  <!-- Logo Section -->
  <div style="background: none; text-align: center; margin: 30px 0 10px;">
    <img src="https://www.cemfi.es/images/Logo-Azul.png" alt="CEMFI Logo" style="width: 158px; filter: drop-shadow(0 2px 12px rgba(56,84,156,0.05)); margin-bottom: 0;">
  </div>

  <!-- Name -->
  <div style="font-family: 'Times New Roman', Times, serif; color: #38549c; text-align: center; font-size: 1.22em; font-weight: bold; margin-bottom: 0px;">
    Jesus Villota Miranda © 2025
  </div>

  <!-- Contact info -->
  <div style="font-family: 'Times New Roman', Times, serif; color: #38549c; text-align: center; font-size: 1em; margin-top: 7px; margin-bottom: 20px;">
    <a href="mailto:jesus.villota@cemfi.edu.es" style="color: #38549c; text-decoration: none; margin-right:8px;" title="Email">
      <!-- <img src="https://cdn-icons-png.flaticon.com/512/11679/11679732.png" alt="Email" style="width:18px; vertical-align:middle; margin-right:5px;"> -->
      jesus.villota@cemfi.edu.es
    </a>
    <span style="color:#9fa7bd;">|</span>
    <a href="https://www.linkedin.com/in/jesusvillotamiranda/" target="_blank" style="color: #38549c; text-decoration: none; margin-left:7px;" title="LinkedIn">
      <!-- <img src="https://1.bp.blogspot.com/-onvhHUdW1Us/YI52e9j4eKI/AAAAAAAAE4c/6s9wzOpIDYcAo4YmTX1Qg51OlwMFmilFACLcBGAsYHQ/s1600/Logo%2BLinkedin.png" alt="LinkedIn" style="width:17px; vertical-align:middle; margin-right:5px;"> -->
      LinkedIn
    </a>
  </div>
</div>

:::
Sources:
- Villota, Jesus, *Structured Data with LLMs Done Right: A Practical Guide to Text Classification, Information Retrieval and Generation with LLMs* (October 21, 2025). Available at SSRN: <https://ssrn.com/abstract=5636430> or <http://dx.doi.org/10.2139/ssrn.5636430>.

- Villota, Jesus, *Predicting Market Reactions to News: An LLM-Based Approach Using Spanish Business Articles* (June 10, 2024). CEMFI Working Paper 2501, Available at SSRN: <https://ssrn.com/abstract=5006857> or <http://dx.doi.org/10.2139/ssrn.5006857>.
:::

---

**IMPORTANT**: **Are you running this notebook in Google Colab?**

- If so, please make sure that in the cell below `running_in_colab` is set to `True`

- And, of course,  make sure to **run the cell**!

In [20]:
running_in_colab = False

## What we'll be doing here (big picture)

 

- Goal: show how “function calling” lets an LLM return structured data by invoking your Python functions with validated arguments.

- We’ll build two small applications:

  1) News → firm-level shocks: from a Spanish article, extract affected Spanish listed firms and classify shock type/magnitude/direction.

  2) Central bank speeches → policy stance classification and economic indicator extraction: classify monetary policy stance (hawkish/dovish/neutral) and extract economic forecasts (GDP, inflation, policy rates) from ECB speeches.

- Core pattern you’ll see in both:

  1) Define a tool schema (name, description, JSON parameters) for the function you want the model to call.

  2) Ask the LLM with tools enabled; if it decides to call your tool, it provides JSON args matching the schema.

  3) Validate/execute your local Python function; return its result to the model as a tool response.

  4) Make a second model call so the LLM can integrate the tool’s structured output into a final answer.

In [21]:
import os
import json
import pandas as pd

if running_in_colab:
    ! pip install groq
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')
else:
    import os
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv('GROQ_API_KEY')

In [22]:
from groq import Groq
client = Groq(api_key=api_key)
MODEL = 'llama-3.3-70b-versatile'

<div style="max-width: 880px; margin: 20px auto 22px; padding: 20px 32px; border-radius: 18px; border: 1px solid #e5e7eb; background: linear-gradient(180deg, #ffffff 0%, #f9fafb 100%); box-shadow: 0 8px 26px rgba(0,0,0,0.06); overflow: hidden; text-align: center;">

  <div style="font-size: 20px; font-weight: 800; color: #14276c; margin-bottom: 8px;">
    Application #1)
  </div>
  <div style="font-size: 18px; font-weight: 700; color: #374151; font-style: italic;">
    Extracting & categorizing news-implied firm-specific shocks
  </div>
  
</div>


This notebook replicates the methodology in [CEMFI Working Paper 2501](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5006857).

In [23]:
def news_parser(firms):
    response = []
    for firm in firms:
        response.append({
            "firm": firm["firm"],
            "ticker": firm.get("ticker", ""),
            "shock_type": firm.get("shock_type", ""),
            "shock_magnitude": firm.get("shock_magnitude", ""),
            "shock_direction": firm.get("shock_direction", ""),
        })
    return response

def run_conversation(user_prompt):
    # Step 1: send the conversation and available functions to the model
    messages = [
        {
            "role": "system",
            "content":  f"""
                            You are a function calling LLM that analyses business news in Spanish. 
                        """
        },
        {
            "role": "user",
            "content": user_prompt,
        }
    ]
    
    tools = [
        {
            "type": "function",
            "function": {
                "name": "news_parser",
                "description": f"""
                                    For every article, you must identify the firms directly affected by the news. Do not include every firm mentioned in the article, only include those that are directly affected by the shocks narrated therein. 
                                    The identified firms must be Spanish and should be publicly listed in the Spanish exchange (their ticker is of the form 'TICKER.MC'). Do not include non-Spanish foreign firms. Do not include Spanish firms that are not publicly traded.
                                    For each identified firm, classify the shocks that affect them (type, magnitude, category). The type of shock can be 'demand', 'supply', 'financial', 'policy', or 'technology'. The magnitude can be 'minor' or 'major'. The direction can be 'positive' or 'negative'.
                                    If a firm is neutral to the article, do NOT include it in the analysis.
                                """,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "firms": {
                            "type": "array",
                            "description": f"""
                                                List the Spanish firms impacted by the reported news. Such firms must be publicly listed in the Spanish stock exchange and have a stock market ticker of the form TICKER.MC. 
                                                Foreign firms (not listed in the Spanish exchange and whose ticker is not TICKER.MC) are not to be included here. Do not include firms that are mentioned just for contextual comparison but are not directly affected by the events described in the article.
                                                If a firm is neutral to the article, do not include it in the list.
                                                Some times the article mentions explicitly the Spanish ticker of those firms that are directly affected (and hence, the firms to include here). e.g: Iberdrola (IBE.MC).
                                            """,
                            "items": {
                                "type": "object",
                                "properties": {
                                    "firm": {
                                        "type": "string",
                                        "description": "State the Spanish firm (within the list 'firms') in which you will focus the analysis. This firm should be publicly traded in the Spanish exchange with a ticker of the form 'TICKER.MC'. ",
                                    },
                                    "ticker": {
                                        "type": "string",
                                        "description": "Specify its stock market ticker of the Spanish firm in Yahoo Finance format (note that Spanish firms' tickers end with '.MC', e.g., ITX.MC for Inditex, ACX.MC for Acerinox, SAN.MC for Banco Santander, NTGY.MC for Naturgy).",
                                    },
                                    "shock_type": {
                                        "type": "string",
                                        "enum": ["demand", "supply", "financial", "policy", "technology"],
                                        "description": "Classify the type of shock implied by the news article. Choose 'demand' for events impacting consumer demand, 'supply' for events affecting the supply of goods or services, 'financial' for events related to financial markets or conditions, 'policy' for events stemming from changes in government policies or regulations, and 'technology' for events resulting from significant technological advancements or disruptions.",
                                    },
                                    "shock_magnitude": {
                                        "type": "string",
                                        "enum": ["minor", "major"],
                                        "description": "How strong do you expect the shock to be: 'minor' or 'major'?",
                                    },
                                    "shock_direction": {
                                        "type": "string",
                                        "enum": ["positive", "negative"],
                                        "description": f"""
                                                        In what direction do you expect the shock to affect this firm? Choose one of the available options: 'positive' or 'negative'.
                                                        Choose 'positive' for beneficial impacts and 'negative' for adverse impacts.
                                                        Do not state 'neutral' here. If the firm is neutral to the article, do not include it in the list of firms.
                                                        """,
                                    },
                                },
                                "required": ["firm"],
                            },
                        },
                    },
                    "required": ["firms"],
                },
            },
        },
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        max_tokens=4096
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # Step 2: check if the model wanted to call a function
    if tool_calls:
        
        # Step 3: call the function
        available_functions = {
            "news_parser": news_parser,
        }
        messages.append(response_message)  # extend conversation with assistant's reply
        
        # Step 4: send the info for each function call and function response to the model
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_call(
                firms=function_args.get("firms")
            )
            messages.append(
                {
                    "role": "function",
                    "name": function_name,
                    "content": json.dumps(function_response),
                }
            )  # extend conversation with function response
        second_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )  # get a new response from the model where it can see the function response
        
        return second_response.choices[0].message.content, function_response


user_prompt = f"""
Cellnex tendrá más competencia en Europa.  La filial de Telefónica (TEF.MC) Telxius Telecom ha acordado vender su división de torres de telecomunicaciones en Europa y Latinoamérica a American Tower (AMT), lo cual aumentará la presencia de ésta en Europa e incrementará la competencia para el grupo español de telecomunicaciones inalámbricas Cellnex Telecom (CLNX.MC), señala Equita Sim. La transacción "supone la entrada de un nuevo operador independiente de torres en el mercado español y potencialmente más competencia para el crecimiento futuro también en el mercado europeo", sostiene la correduría. Cellnex llegó a un acuerdo en noviembre con CK Hutchison (0001.HK) para comprar el negocio europeo de torres y sus activos del conglomerado cotizado en Hong Kong. La acción de Telefónica sube un 9,6% a EUR3,94 y la de Cellnex avanza un 0,3% a EUR47,79.
"""

completion_text, structured_output = run_conversation(user_prompt)
print("Completion Text:", completion_text)
print("Structured Output:", structured_output)


Completion Text: El anuncio de que Telxius Telecom, filial de Telefónica, va a vender su división de torres de telecomunicaciones en Europa y Latinoamérica a American Tower podría aumentar la competencia en Europa para Cellnex Telecom. Esto podría tener un impacto negativo en el crecimiento y las ganancias de Cellnex en el futuro. 

Por otro lado, la transacción tiene un impacto positivo en las acciones de Telefónica, subiendo un 9,6% a EUR3,94. Mientras que la acción de Cellnex solo avanza un 0,3% a EUR47,79. 

En resumen, la noticia tiene un impacto negativo en Cellnex Telecom y un impacto positivo en Telefónica.
Structured Output: [{'firm': 'Cellnex Telecom', 'ticker': 'CLNX.MC', 'shock_type': 'supply', 'shock_magnitude': 'major', 'shock_direction': 'negative'}, {'firm': 'Telefónica', 'ticker': 'TEF.MC', 'shock_type': 'financial', 'shock_magnitude': 'major', 'shock_direction': 'positive'}]


In [24]:
structured_output

[{'firm': 'Cellnex Telecom',
  'ticker': 'CLNX.MC',
  'shock_type': 'supply',
  'shock_magnitude': 'major',
  'shock_direction': 'negative'},
 {'firm': 'Telefónica',
  'ticker': 'TEF.MC',
  'shock_type': 'financial',
  'shock_magnitude': 'major',
  'shock_direction': 'positive'}]

<div style="max-width: 880px; margin: 20px auto 22px; padding: 20px 32px; border-radius: 18px; border: 1px solid #e5e7eb; background: linear-gradient(180deg, #ffffff 0%, #f9fafb 100%); box-shadow: 0 8px 26px rgba(0,0,0,0.06); overflow: hidden; text-align: center;">

  <div style="font-size: 20px; font-weight: 800; color: #14276c; margin-bottom: 8px;">
    Application #2
  </div>
  <div style="font-size: 18px; font-weight: 700; color: #374151; font-style: italic;">
    Classifying Central Bank Speeches & Extracting Economic Indicators
  </div>
  
</div>


In [25]:
# Load ECB speeches dataset
from pathlib import Path

if running_in_colab:
    data_path = Path('/content/gdrive/My Drive/data/all_ECB_speeches.csv')
else:
    data_path = Path('../data/all_ECB_speeches.csv')

# Load CSV file (pipe-delimited)
df_ecb = pd.read_csv(data_path, sep='|', encoding='utf-8')
df_ecb.dropna(subset=['contents'], inplace=True)

print(f"✅ Loaded dataset with {len(df_ecb):,} speeches")
print(f"\nDataset columns: {list(df_ecb.columns)}")
print(f"\nFirst few rows:")
display(df_ecb.head())

# Display basic statistics
print(f"\n{'='*60}")
print("Dataset Statistics:")
print(f"{'='*60}")
print(f"Total number of speeches: {len(df_ecb):,}")
print(f"Unique speakers: {df_ecb['speakers'].nunique() if 'speakers' in df_ecb.columns else 'N/A'}")
if 'contents' in df_ecb.columns:
    print(f"Average speech length: {df_ecb['contents'].str.len().mean():.0f} characters")
    print(f"Median speech length: {df_ecb['contents'].str.len().median():.0f} characters")

✅ Loaded dataset with 2,431 speeches

Dataset columns: ['date', 'speakers', 'title', 'subtitle', 'contents']

First few rows:


,date,speakers,title,subtitle,contents
1,2020-12-16,Isabel Schnabel,The importance of trust for the ECB’s monetary...,"Speech by Isabel Schnabel, Member of the Execu...",SPEECH The importance of trust for the ECB’s...
2,2020-12-16,Fabio Panetta,Keeping cyber risk at bay: our individual and ...,"Introductory remarks by Fabio Panetta, Member ...",SPEECH Keeping cyber risk at bay: our indivi...
4,2020-12-14,Isabel Schnabel,Welcome address,"Welcome address by Isabel Schnabel, Member of ...",SPEECH Welcome address Welcome address by Is...
5,2020-12-14,Fabio Panetta,A commitment to the recovery,"Speech by Fabio Panetta, Member of the Executi...",SPEECH A commitment to the recovery Speech...
6,2020-11-27,Fabio Panetta,From the payments revolution to the reinventio...,"Speech by Fabio Panetta, Member of the Executi...",SPEECH From the payments revolution to the ...



Dataset Statistics:
Total number of speeches: 2,431
Unique speakers: 26
Average speech length: 19685 characters
Median speech length: 17620 characters


## Function Calling: Policy Stance Classification

We define a function-calling schema to classify ECB speeches according to their monetary policy stance. For each speech, we classify whether it expresses: (a) hawkish stance (preference for raising rates, inflation concerns), (b) dovish stance (preference for lowering rates, growth concerns), or (c) neutral stance (balanced view, no clear direction). The tool returns a structured output with the policy stance classification, confidence score, and reasoning.


In [26]:
def classify_speech_stance(stance_data):
    """
    Parser function for monetary policy stance classification.
    Validates and returns structured output.
    """
    return {
        "policy_stance": stance_data.get("policy_stance", ""),
        "confidence": stance_data.get("confidence", 0.0),
        "reasoning": stance_data.get("reasoning", "")
    }


def run_stance_classification(speech_text):
    """
    Classify a central bank speech according to monetary policy stance.
    Returns both completion text and structured output.
    """
    # Check if speech content is valid
    if not speech_text or pd.isna(speech_text) or (isinstance(speech_text, str) and len(speech_text.strip()) == 0):
        return "Cannot classify speech: speech content is empty or missing.", None
    
    messages = [
        {
            "role": "system",
            "content": "You are an expert in monetary policy and central bank communication analysis."
        },
        {
            "role": "user",
            "content": f"Classify the following central bank speech according to its monetary policy stance:\n\n{speech_text}"
        }
    ]
    
    tools = [
        {
            "type": "function",
            "function": {
                "name": "classify_monetary_policy_stance",
                "description": "Classify central bank communication based on policy stance",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "policy_stance": {
                            "type": "string",
                            "enum": ["hawkish", "dovish", "neutral"],
                            "description": """Classify the text as:
                            - 'hawkish': preference for raising interest rates, 
                              concerns about inflation, tightening monetary policy
                            - 'dovish': preference for lowering rates, concerns about 
                              unemployment/growth, easing monetary policy
                            - 'neutral': balanced view or no clear policy direction"""
                        },
                        "confidence": {
                            "type": "number",
                            "minimum": 0,
                            "maximum": 1,
                            "description": "Confidence score for the classification"
                        },
                        "reasoning": {
                            "type": "string",
                            "description": "Brief explanation of the classification"
                        }
                    },
                    "required": ["policy_stance", "confidence", "reasoning"],
                    "additionalProperties": False
                }
            }
        }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        max_tokens=4096
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # Step 2: check if the model wanted to call a function
    if tool_calls:
        # Step 3: call the function
        available_functions = {
            "classify_monetary_policy_stance": classify_speech_stance,
        }
        messages.append(response_message)  # extend conversation with assistant's reply
        
        # Step 4: send the info for each function call and function response to the model
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_call(stance_data=function_args)
            messages.append(
                {
                    "role": "function",
                    "name": function_name,
                    "content": json.dumps(function_response),
                }
            )  # extend conversation with function response
        second_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )  # get a new response from the model where it can see the function response
        
        return second_response.choices[0].message.content, function_response
    
    # If no tool call, return the plain response
    return response_message.content, None


# Classify sample speech 1
print("Classifying Sample Speech...")
completion_text, structured_output = run_stance_classification(df_ecb.contents.iloc[0])

print("\n" + "="*60)
print("Completion Text:")
print("="*60)
print(completion_text)

print("\n" + "="*60)
print("Structured Output:")
print("="*60)
print(structured_output)

Classifying Sample Speech...

Completion Text:
The classification of the central bank speech is: 

**Monetary Policy Stance:** Dovish
**Confidence:** 0.8
**Reasoning:** The speech by Isabel Schnabel, Member of the Executive Board of the ECB, emphasizes the importance of trust in the ECB and the euro, and highlights the ECB's efforts to support the economy during the COVID-19 pandemic. The tone of the speech is cautious and supportive, indicating a dovish monetary policy stance. The speech also mentions the ECB's measures to ensure favorable financing conditions and ample liquidity provision, which is consistent with a dovish stance.

Structured Output:
{'policy_stance': 'dovish', 'confidence': 0.8, 'reasoning': "The speech by Isabel Schnabel, Member of the Executive Board of the ECB, emphasizes the importance of trust in the ECB and the euro, and highlights the ECB's efforts to support the economy during the COVID-19 pandemic. The tone of the speech is cautious and supportive, indicati

In [27]:
structured_output

{'policy_stance': 'dovish',
 'confidence': 0.8,
 'reasoning': "The speech by Isabel Schnabel, Member of the Executive Board of the ECB, emphasizes the importance of trust in the ECB and the euro, and highlights the ECB's efforts to support the economy during the COVID-19 pandemic. The tone of the speech is cautious and supportive, indicating a dovish monetary policy stance. The speech also mentions the ECB's measures to ensure favorable financing conditions and ample liquidity provision, which is consistent with a dovish stance."}

## Task 2: Extracting Economic Indicators

We now define a function-calling schema to extract specific economic forecasts and policy decisions from central bank speeches. This information retrieval task extracts structured numerical data (GDP forecasts, inflation forecasts, policy rate changes) from unstructured text, enabling systematic analysis of central bank communications.


In [28]:
def extract_economic_indicators_parser(indicators_data):
    """
    Parser function for economic indicator extraction.
    Validates and returns structured output.
    """
    return {
        "gdp_forecast": indicators_data.get("gdp_forecast"),
        "inflation_forecast": indicators_data.get("inflation_forecast"),
        "unemployment_forecast": indicators_data.get("unemployment_forecast"),
        "policy_rate_change": indicators_data.get("policy_rate_change", 0),
        "forecast_horizon": indicators_data.get("forecast_horizon", ""),
        "confidence": indicators_data.get("confidence", 0.0)
    }


def run_economic_extraction(speech_text):
    """
    Extract economic indicators from a central bank speech.
    Returns both completion text and structured output.
    """
    # Check if speech content is valid
    if not speech_text or pd.isna(speech_text) or (isinstance(speech_text, str) and len(speech_text.strip()) == 0):
        return "Cannot extract economic indicators: speech content is empty or missing.", None
    
    messages = [
        {
            "role": "system",
            "content": "You are an expert in extracting economic forecasts and policy decisions from central bank communications."
        },
        {
            "role": "user",
            "content": f"Extract economic indicators and policy decisions from the following central bank speech:\n\n{speech_text}"
        }
    ]
    
    tools = [
        {
            "type": "function",
            "function": {
                "name": "extract_economic_indicators",
                "description": "Extract economic forecasts and policy decisions from central bank text",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "gdp_forecast": {
                            "type": "number",
                            "description": "GDP growth forecast (percentage). Use null if not mentioned."
                        },
                        "inflation_forecast": {
                            "type": "number",
                            "description": "Inflation forecast (percentage). Use null if not mentioned."
                        },
                        "unemployment_forecast": {
                            "type": "number",
                            "description": "Unemployment rate forecast (percentage). Use null if not mentioned."
                        },
                        "policy_rate_change": {
                            "type": "number",
                            "description": "Policy rate change in basis points (e.g., 25 for 0.25%). Use 0 if no change."
                        },
                        "forecast_horizon": {
                            "type": "string",
                            "description": "Time horizon for forecasts (e.g., '2024 Q4', '2025', 'current year')"
                        },
                        "confidence": {
                            "type": "number",
                            "minimum": 0,
                            "maximum": 1,
                            "description": "Confidence in the extracted information"
                        }
                    },
                    "required": ["policy_rate_change", "forecast_horizon", "confidence"],
                    "additionalProperties": False
                }
            }
        }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        max_tokens=4096
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # Step 2: check if the model wanted to call a function
    if tool_calls:
        # Step 3: call the function
        available_functions = {
            "extract_economic_indicators": extract_economic_indicators_parser,
        }
        messages.append(response_message)  # extend conversation with assistant's reply
        
        # Step 4: send the info for each function call and function response to the model
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_call(indicators_data=function_args)
            messages.append(
                {
                    "role": "function",
                    "name": function_name,
                    "content": json.dumps(function_response),
                }
            )  # extend conversation with function response
        second_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )  # get a new response from the model where it can see the function response
        
        return second_response.choices[0].message.content, function_response
    
    # If no tool call, return the plain response
    return response_message.content, None


# Extract economic indicators

print("Extracting Economic Indicators...")
completion_text, structured_output = run_economic_extraction(df_ecb.contents.iloc[0])

print("\n" + "="*60)
print("Completion Text:")
print("="*60)
print(completion_text)

print("\n" + "="*60)
print("Structured Output:")
print("="*60)
print(structured_output)


Extracting Economic Indicators...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hw97bchyeb5a05mjaahdvtzf` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98769, Requested 8151. Please try again in 1h39m38.88s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}